# Train subset

Verify the dataset

In [1]:
import torch

data = torch.load("fsl_dataset_clean.pt")

X = data["X"]
y = data["y"]

print(X.shape)
print(y.shape)

torch.Size([2129, 30, 126])
torch.Size([2129])


In [4]:
import torch

data = torch.load("fsl_dataset_clean.pt")

X = data["X"]

bad = torch.where(
    torch.all(X == 0, dim=(1, 2))
)[0]

print(bad)

tensor([], dtype=torch.int64)


In [7]:
labels, counts = torch.unique(y, return_counts=True)

for label, count in zip(labels.tolist(), counts.tolist()):
    print(f"Class {label}: {count} samples")

Class 0: 20 samples
Class 1: 21 samples
Class 2: 22 samples
Class 3: 20 samples
Class 4: 21 samples
Class 5: 20 samples
Class 6: 22 samples
Class 7: 20 samples
Class 8: 20 samples
Class 9: 20 samples
Class 10: 20 samples
Class 11: 21 samples
Class 12: 21 samples
Class 13: 20 samples
Class 14: 21 samples
Class 15: 20 samples
Class 16: 21 samples
Class 17: 22 samples
Class 18: 21 samples
Class 19: 21 samples
Class 20: 20 samples
Class 21: 20 samples
Class 22: 20 samples
Class 23: 20 samples
Class 24: 20 samples
Class 25: 20 samples
Class 26: 21 samples
Class 27: 20 samples
Class 28: 20 samples
Class 29: 20 samples
Class 30: 20 samples
Class 31: 19 samples
Class 32: 20 samples
Class 33: 20 samples
Class 34: 21 samples
Class 35: 20 samples
Class 36: 21 samples
Class 37: 22 samples
Class 38: 20 samples
Class 39: 20 samples
Class 40: 22 samples
Class 41: 22 samples
Class 42: 20 samples
Class 43: 20 samples
Class 44: 20 samples
Class 45: 20 samples
Class 46: 20 samples
Class 47: 20 samples
Cl

In [ ]:
selected_classes = [0, 1, 2, 3, 4]

mask = torch.isin(
    y,
    torch.tensor(selected_classes)
)

X_small = X[mask]
y_small = y[mask]

print(X_small.shape)
print(y_small.shape)

print(X_greetings.shape)
print(y_greetings.shape)

unique, counts = np.unique(y_greetings, return_counts=True)

for label, count in zip(unique, counts):
    print(label, count)

torch.Size([104, 30, 126])
torch.Size([104])


In [5]:
print(torch.unique(y_small, return_counts=True))


(tensor([0, 1, 2, 3, 4]), tensor([20, 21, 22, 20, 21]))


Remap labels

In [6]:
selected_classes = sorted(torch.unique(y_small).tolist())

label_map = {
    old: new
    for new, old in enumerate(selected_classes)
}

y_small = torch.tensor(
    [label_map[int(label)] for label in y_small]
)

print(label_map)



{0: 0, 1: 1, 2: 2, 3: 3, 4: 4}


Train/Validation/Test split from scikit learn

In [9]:
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

# -------------------------
# Train / Validation / Test
# 70% / 15% / 15%
# -------------------------

# First split: 70% train, 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X_small,
    y_small,
    test_size=0.30,
    stratify=y_small,
    random_state=42,
)

# Second split: 30% -> 15% validation + 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42,
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

# -------------------------
# PyTorch datasets
# -------------------------

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)


Train: torch.Size([72, 30, 126])
Validation: torch.Size([16, 30, 126])
Test: torch.Size([16, 30, 126])


In [10]:
print(type(X_train))
print(type(y_train))

<class 'torch.Tensor'>
<class 'torch.Tensor'>


In [ ]:

# -------------------------
# DataLoaders
# -------------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
)

In [12]:
for batch_X, batch_y in train_loader:
    print("Batch X:", batch_X.shape)
    print("Batch y:", batch_y.shape)
    break

Batch X: torch.Size([32, 30, 126])
Batch y: torch.Size([32])


In [13]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)

cpu


Cell 10 — Create your first model

Start simple.

For dynamic sign recognition, an LSTM is a good baseline.

Architecture:

(30,126)
    |
    ↓
 LSTM
    |
    ↓
 Fully Connected
    |
    ↓
 Class prediction

In [14]:
import torch.nn as nn


class SignLSTM(nn.Module):

    def __init__(
        self,
        input_size=126,
        hidden_size=128,
        num_classes=5,
        num_layers=2,
        dropout=0.3,
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,
        )

        self.fc = nn.Linear(
            hidden_size,
            num_classes
        )


    def forward(self, x):

        output, (hidden, cell) = self.lstm(x)

        # take last timestep
        x = hidden[-1]

        x = self.fc(x)

        return x


Initialize Model

In [15]:
num_classes = len(torch.unique(y_small))

model = SignLSTM(
    num_classes=num_classes
)

model = model.to(device)

print(model)

SignLSTM(
  (lstm): LSTM(126, 128, num_layers=2, batch_first=True, dropout=0.3)
  (fc): Linear(in_features=128, out_features=5, bias=True)
)


Define loss and optimizer

In [16]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

Training function

In [17]:
def train_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device
):

    model.train()

    total_loss = 0
    correct = 0
    total = 0


    for X_batch, y_batch in loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)


        optimizer.zero_grad()


        outputs = model(X_batch)

        loss = criterion(
            outputs,
            y_batch
        )


        loss.backward()

        optimizer.step()


        total_loss += loss.item()


        predictions = torch.argmax(
            outputs,
            dim=1
        )

        correct += (
            predictions == y_batch
        ).sum().item()

        total += y_batch.size(0)


    accuracy = correct / total

    return (
        total_loss / len(loader),
        accuracy
    )

Validation function

In [18]:
def evaluate(
    model,
    loader,
    criterion,
    device
):

    model.eval()

    total_loss = 0
    correct = 0
    total = 0


    with torch.no_grad():

        for X_batch, y_batch in loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)


            outputs = model(X_batch)

            loss = criterion(
                outputs,
                y_batch
            )


            total_loss += loss.item()


            predictions = torch.argmax(
                outputs,
                dim=1
            )

            correct += (
                predictions == y_batch
            ).sum().item()

            total += y_batch.size(0)


    accuracy = correct / total


    return (
        total_loss / len(loader),
        accuracy
    )


Train loop

In [19]:
epochs = 30


for epoch in range(epochs):

    train_loss, train_acc = train_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device
    )


    val_loss, val_acc = evaluate(
        model,
        val_loader,
        criterion,
        device
    )


    print(
        f"""
        Epoch {epoch+1}/{epochs}

        Train Loss: {train_loss:.4f}
        Train Acc: {train_acc:.4f}

        Val Loss: {val_loss:.4f}
        Val Acc: {val_acc:.4f}
        """
    )


        Epoch 1/30

        Train Loss: 1.6049
        Train Acc: 0.1944

        Val Loss: 1.5911
        Val Acc: 0.2500
        

        Epoch 2/30

        Train Loss: 1.5970
        Train Acc: 0.2778

        Val Loss: 1.5675
        Val Acc: 0.3750
        

        Epoch 3/30

        Train Loss: 1.5519
        Train Acc: 0.4028

        Val Loss: 1.5161
        Val Acc: 0.5625
        

        Epoch 4/30

        Train Loss: 1.4848
        Train Acc: 0.5278

        Val Loss: 1.3822
        Val Acc: 0.6250
        

        Epoch 5/30

        Train Loss: 1.3317
        Train Acc: 0.6528

        Val Loss: 1.1818
        Val Acc: 0.5625
        

        Epoch 6/30

        Train Loss: 1.1112
        Train Acc: 0.6389

        Val Loss: 1.0174
        Val Acc: 0.5000
        

        Epoch 7/30

        Train Loss: 0.9844
        Train Acc: 0.5972

        Val Loss: 0.8341
        Val Acc: 0.6250
        

        Epoch 8/30

        Train Loss: 0.7690
        Train Acc: 0.

Save the best model (do this first)

In [20]:
best_val_acc = 0

for epoch in range(epochs):

    train_loss, train_acc = train_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device
    )


    val_loss, val_acc = evaluate(
        model,
        val_loader,
        criterion,
        device
    )


    if val_acc > best_val_acc:
        best_val_acc = val_acc

        torch.save(
            model.state_dict(),
            "best_sign_lstm.pt"
        )

        print("Saved best model!")

Saved best model!
Saved best model!


In [21]:
model.load_state_dict(
    torch.load("best_sign_lstm.pt")
)

model.eval()

SignLSTM(
  (lstm): LSTM(126, 128, num_layers=2, batch_first=True, dropout=0.3)
  (fc): Linear(in_features=128, out_features=5, bias=True)
)

In [22]:
test_loss, test_acc = evaluate(
    model,
    test_loader,
    criterion,
    device
)

print("Test accuracy:", test_acc)

Test accuracy: 0.625


In [23]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt


all_preds = []
all_labels = []


model.eval()

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        X_batch = X_batch.to(device)

        outputs = model(X_batch)

        preds = torch.argmax(
            outputs,
            dim=1
        )


        all_preds.extend(
            preds.cpu().numpy()
        )

        all_labels.extend(
            y_batch.numpy()
        )


cm = confusion_matrix(
    all_labels,
    all_preds
)

print(cm)

[[2 0 1 0 0]
 [0 1 2 0 0]
 [1 2 1 0 0]
 [0 0 0 3 0]
 [0 0 0 0 3]]


In [24]:
print(torch.unique(
    y_small,
    return_counts=True
))

(tensor([0, 1, 2, 3, 4]), tensor([20, 21, 22, 20, 21]))


cell that maps label IDs back to names:

In [6]:
import pandas as pd

manifest = pd.read_csv(r"C:\Projects\signia-fsl-recognition\data\video_manifest.csv")

lookup = (
    manifest[["label_id", "display_label"]]
    .drop_duplicates()
    .sort_values("label_id")
)

print(lookup)

      label_id   display_label
0            0    Good Morning
20           1  Good Afternoon
41           2    Good Evening
63           3           Hello
83           4     How Are You
...        ...             ...
2029       100             Tea
2049       101            Beer
2069       102            Wine
2089       103           Sugar
2110       104        No Sugar

[105 rows x 2 columns]


In [1]:
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader


data = torch.load("fsl_dataset_cleaned.pt")

X = data["X"]
y = data["y"]


labels_df = pd.read_csv("csv/expanded_labels.csv")


selected_classes = [0, 1, 2, 3, 4]



FileNotFoundError: [Errno 2] No such file or directory: 'fsl_dataset_cleaned.pt'